# 03 — Baseline Models
**ZivaBasa MVP (Kaggle-Data Phase)**

Trains classical ML baselines — Logistic/Linear Regression, Decision Tree, Random Forest,
Gradient Boosting — **one set per task head** (Employment, Skills, Productivity), using the
processed features from `02_feature_engineering.ipynb`.

**Purpose:** these are the empirical bar the multi-task neural network (notebook 04) has to
clear. If the deep model doesn't beat these, that's a real finding to report, not something to
bury.

- Employment → classification (`target_high_automation_risk`)
- Skills → classification (`target_attrition`)
- Productivity → regression (`target_ai_adoption`)

All runs are logged to MLflow so today's numbers are auditable later.


In [ ]:
# --- Setup ---
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.ensemble import GradientBoostingClassifier, GradientBoostingRegressor
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
    mean_squared_error, mean_absolute_error, r2_score, confusion_matrix
)

import mlflow

pd.set_option("display.max_columns", 100)
sns.set_theme(style="whitegrid")

PROCESSED_DIR = "../data/processed"
MODELS_DIR = "../models"
os.makedirs(MODELS_DIR, exist_ok=True)

mlflow.set_tracking_uri("../mlruns")
mlflow.set_experiment("zivabasa_mvp_baselines")

RANDOM_STATE = 42


## 1. Load Processed Features


In [ ]:
def load_features(name):
    path = os.path.join(PROCESSED_DIR, f"{name}_features.parquet")
    if not os.path.exists(path):
        print(f"[MISSING] {path} — run 02_feature_engineering.ipynb first.")
        return None
    df = pd.read_parquet(path)
    print(f"[{name}] loaded {df.shape[0]:,} rows x {df.shape[1]} cols")
    return df

feat_employment = load_features("employment")
feat_skills = load_features("skills")
feat_productivity = load_features("productivity")


## 2. Task Configuration

One config block per task head: target column, task type, and which columns to drop from the
feature matrix (targets, ID-like columns, anything that would leak the label).


In [ ]:
TASK_CONFIG = {
    "employment": {
        "df": feat_employment,
        "target": "target_high_automation_risk",
        "task_type": "classification",
        "drop_cols": ["target_high_automation_risk", "automation_risk", "automation_exposure_index"],
    },
    "skills": {
        "df": feat_skills,
        "target": "target_attrition",
        "task_type": "classification",
        "drop_cols": ["target_attrition"],
    },
    "productivity": {
        "df": feat_productivity,
        "target": "target_ai_adoption",
        "task_type": "regression",
        "drop_cols": ["target_ai_adoption", "ai_adoption_level", "ai_adoption_index"],
    },
}

for name, cfg in TASK_CONFIG.items():
    df = cfg["df"]
    if df is None:
        print(f"[{name}] no data loaded — skipping.")
        continue
    if cfg["target"] not in df.columns:
        print(f"[{name}] WARNING — target '{cfg['target']}' not found. Re-check notebook 02, Section 9.")
    else:
        print(f"[{name}] target OK: '{cfg['target']}' ({cfg['task_type']})")


## 3. Train / Test Split

80/20 split per task head, stratified on the target for classification tasks. Each task head is
split independently since the datasets aren't the same population (see README, Section 7).


In [ ]:
splits = {}

for name, cfg in TASK_CONFIG.items():
    df = cfg["df"]
    if df is None or cfg["target"] not in df.columns:
        continue

    drop_cols = [c for c in cfg["drop_cols"] if c in df.columns]
    X = df.drop(columns=drop_cols)
    X = X.select_dtypes(include=[np.number])  # safety net: drop any leftover non-numeric cols
    y = df[cfg["target"]]

    stratify = y if cfg["task_type"] == "classification" else None
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=stratify
    )
    splits[name] = {"X_train": X_train, "X_test": X_test, "y_train": y_train, "y_test": y_test}
    print(f"[{name}] train={X_train.shape[0]:,} test={X_test.shape[0]:,} features={X_train.shape[1]}")


## 4. Model Zoo

Same four model types per task head, swapped between classifier/regressor variants based on
`task_type`.


In [ ]:
def get_models(task_type):
    if task_type == "classification":
        return {
            "logistic_regression": LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
            "decision_tree": DecisionTreeClassifier(max_depth=8, random_state=RANDOM_STATE),
            "random_forest": RandomForestClassifier(n_estimators=200, max_depth=10, random_state=RANDOM_STATE),
            "gradient_boosting": GradientBoostingClassifier(n_estimators=200, max_depth=3, random_state=RANDOM_STATE),
        }
    else:
        return {
            "linear_regression": LinearRegression(),
            "decision_tree": DecisionTreeRegressor(max_depth=8, random_state=RANDOM_STATE),
            "random_forest": RandomForestRegressor(n_estimators=200, max_depth=10, random_state=RANDOM_STATE),
            "gradient_boosting": GradientBoostingRegressor(n_estimators=200, max_depth=3, random_state=RANDOM_STATE),
        }


## 5. Evaluation Functions

Classification → Accuracy, Precision, Recall, F1, ROC-AUC.
Regression → RMSE, MAE, R².


In [ ]:
def evaluate_classification(y_true, y_pred, y_proba=None):
    metrics = {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
    }
    if y_proba is not None:
        try:
            metrics["roc_auc"] = roc_auc_score(y_true, y_proba)
        except ValueError:
            metrics["roc_auc"] = np.nan
    return metrics

def evaluate_regression(y_true, y_pred):
    return {
        "rmse": np.sqrt(mean_squared_error(y_true, y_pred)),
        "mae": mean_absolute_error(y_true, y_pred),
        "r2": r2_score(y_true, y_pred),
    }


## 6. Train + Evaluate + Log to MLflow

Trains every model in the zoo for every task head, evaluates on the held-out test set, and logs
params/metrics/model artifact to MLflow under one run per (task_head, model_name) combination.


In [ ]:
results = []

for task_name, cfg in TASK_CONFIG.items():
    if task_name not in splits:
        print(f"[{task_name}] skipped — no split available.")
        continue

    X_train, X_test = splits[task_name]["X_train"], splits[task_name]["X_test"]
    y_train, y_test = splits[task_name]["y_train"], splits[task_name]["y_test"]
    task_type = cfg["task_type"]
    models = get_models(task_type)

    for model_name, model in models.items():
        run_name = f"{task_name}_{model_name}"
        with mlflow.start_run(run_name=run_name):
            mlflow.log_param("task_head", task_name)
            mlflow.log_param("model_type", model_name)
            mlflow.log_param("task_type", task_type)
            mlflow.log_param("n_features", X_train.shape[1])
            mlflow.log_param("n_train", X_train.shape[0])
            mlflow.log_param("n_test", X_test.shape[0])

            model.fit(X_train, y_train)
            y_pred = model.predict(X_test)

            if task_type == "classification":
                y_proba = model.predict_proba(X_test)[:, 1] if hasattr(model, "predict_proba") else None
                metrics = evaluate_classification(y_test, y_pred, y_proba)
            else:
                metrics = evaluate_regression(y_test, y_pred)

            mlflow.log_metrics({k: v for k, v in metrics.items() if not np.isnan(v)})
            mlflow.sklearn.log_model(model, artifact_path="model")

            row = {"task_head": task_name, "model": model_name, **metrics}
            results.append(row)
            print(f"[{run_name}] done -> {metrics}")

results_df = pd.DataFrame(results)
results_df


## 7. Results Comparison

Side-by-side comparison per task head, best model highlighted. This is the empirical baseline
table that goes straight into your evaluation report / thesis chapter.


In [ ]:
for task_name in TASK_CONFIG:
    task_results = results_df[results_df["task_head"] == task_name]
    if task_results.empty:
        continue
    print(f"=== {task_name.upper()} ===")
    sort_col = "roc_auc" if "roc_auc" in task_results.columns and task_results["roc_auc"].notna().any() else \
               ("r2" if "r2" in task_results.columns else task_results.columns[-1])
    display(task_results.sort_values(sort_col, ascending=False).reset_index(drop=True))
    print()


In [ ]:
# Visual comparison
fig, axes = plt.subplots(1, len(TASK_CONFIG), figsize=(6 * len(TASK_CONFIG), 5))
axes = np.array(axes).reshape(-1)

for i, task_name in enumerate(TASK_CONFIG):
    task_results = results_df[results_df["task_head"] == task_name]
    if task_results.empty:
        axes[i].axis("off")
        continue
    metric_col = "roc_auc" if "roc_auc" in task_results.columns and task_results["roc_auc"].notna().any() else "r2"
    sns.barplot(data=task_results, x="model", y=metric_col, ax=axes[i])
    axes[i].set_title(f"{task_name} — {metric_col}")
    axes[i].tick_params(axis="x", rotation=30)

plt.tight_layout()
plt.show()


## 8. Feature Importance (Best Model per Task Head)

Tree-based feature importances give an early, cheap read on which engineered features matter —
useful context before SHAP runs on the neural network in notebook 05.


In [ ]:
def plot_feature_importance(model, feature_names, task_name, model_name, top_n=15):
    if not hasattr(model, "feature_importances_"):
        print(f"[{task_name}/{model_name}] no feature_importances_ attribute, skipping.")
        return
    importances = pd.Series(model.feature_importances_, index=feature_names).sort_values(ascending=False)
    plt.figure(figsize=(8, 6))
    importances.head(top_n).plot(kind="barh")
    plt.gca().invert_yaxis()
    plt.title(f"{task_name} — {model_name} — Top {top_n} Feature Importances")
    plt.tight_layout()
    plt.show()

# Re-fit each task's best tree-based model on full training data for inspection
for task_name, cfg in TASK_CONFIG.items():
    if task_name not in splits:
        continue
    task_results = results_df[results_df["task_head"] == task_name]
    if task_results.empty:
        continue
    sort_col = "roc_auc" if "roc_auc" in task_results.columns and task_results["roc_auc"].notna().any() else "r2"
    tree_results = task_results[task_results["model"].isin(["random_forest", "gradient_boosting", "decision_tree"])]
    if tree_results.empty:
        continue
    best_model_name = tree_results.sort_values(sort_col, ascending=False).iloc[0]["model"]

    models = get_models(cfg["task_type"])
    best_model = models[best_model_name]
    best_model.fit(splits[task_name]["X_train"], splits[task_name]["y_train"])
    plot_feature_importance(best_model, splits[task_name]["X_train"].columns, task_name, best_model_name)


## 9. Save Results Table


In [ ]:
results_path = os.path.join(MODELS_DIR, "baseline_results.csv")
results_df.to_csv(results_path, index=False)
print(f"Baseline results saved -> {results_path}")

print("""
View the full MLflow run comparison with:
    mlflow ui --backend-store-uri ../mlruns
then open http://localhost:5000
""")


## 10. Summary — Carry Forward to Notebook 04

- [ ] Best baseline metric noted per task head (this is the bar the multi-task NN must clear)
- [ ] Any task head with suspiciously perfect scores (accuracy/R² near 1.0) investigated for
      target leakage before moving on
- [ ] Class imbalance noted for classification tasks (check `y_train.value_counts()`) — may need
      `class_weight="balanced"` or resampling in the neural network
- [ ] Feature importances reviewed — do they make domain sense, or hint at a data quality issue?
